# Semanal 2 · GPU: redes semanais e próximos passos da temporada

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem `dados/`, `dados_uf/`, `referencias_imdc3/`, `v4_lib.py`, `comp_lib.py`, `comp_modelos.py`, `uf_lib.py`):
1. suba **`semanal_lib.py`** (pasta `data_imdc_2026/SEMANAL/`);
2. **crie à mão** as pastas **`resultados_semanal`** e **`resultados_uf_prox`** no Drive (assim, duas sessões abertas ao mesmo tempo não criam cópias com o mesmo nome);
3. (opcional, para o Felipe na temporada) a pasta `COMPARACAO_FELIPE/` com `felipe_no_meu_jogo_previsoes.parquet`.

**Retomada:** cada modelo × rodada vira um arquivo assim que termina (gravado num `.tmp` e renomeado, então uma queda no meio nunca deixa arquivo pela metade). Se a sessão cair (internet, energia, limite do Colab), **rode de novo todas as células**: o que já existe é pulado. A seleção de variáveis também grava o estado a cada passo.

| Parte | O que faz | Tempo (GPU L4) |
|---|---|---|
| A · grafos semanais | MTGNN, A3T-GCN, GCN-GRU, STGCN (H=4); MTGNN com H=68 (= o modelo de temporada usado na semana seguinte); MTGNN com alvo em crescimento; MTGNN com todas as covariáveis e com as selecionadas | 1–2 h |
| B · NHITS, TFT, Chronos-2 semanais | sem covariáveis, com todas e com as selecionadas; Chronos-2 zero-shot | 2–3 h |
| C · temporada, próximos passos | MTGNN v2: alvo em crescimento, alvo misto e perda ponderada pela população (raiz), nas 8 temporadas → `resultados_uf_prox` | 2–3 h |

Os modelos `_sel` usam a seleção do notebook 1; se ela ainda não existir, eles são pulados (rode de novo depois).


In [ ]:
NOTEBOOK = 'semanal_gpu'
PACOTES = 'neuralforecast "chronos-forecasting>=2.0"'

# @title 1 · Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/sem', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'dados', 'dados_uf',
                 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    if PACOTES:
        r = subprocess.run(f'pip install -q {PACOTES}', shell=True, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:])
sys.path.insert(0, PASTA_LOCAL)
PASTA_SEM = os.path.join(PASTA_DRIVE, 'resultados_semanal')       # previsões semanais (1 arquivo por modelo × rodada)
PASTA_PROX = os.path.join(PASTA_DRIVE, 'resultados_uf_prox')      # próximos passos da temporada (lida pelo CompUF4/Comp5)
PASTA_AVAL = os.path.join(PASTA_DRIVE, 'resultados_semanal_avaliacao')
PASTA_RES = os.path.join(PASTA_DRIVE, 'resultados_uf')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
PASTA_EXT = os.path.join(PASTA_DRIVE, 'resultados_externos')
for p in (PASTA_SEM, PASTA_PROX, PASTA_AVAL):
    os.makedirs(p, exist_ok=True)
ARQ_SEL = os.path.join(PASTA_SEM, 'selecao_grupos.json')
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U, semanal_lib as S
for m in (V, L, CM, U, S):
    importlib.reload(m)
try:
    import torch
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma', '· CPUs:', os.cpu_count())
except ImportError:
    torch = None
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
TUNING, TESTE = U.TUNING, U.TESTE
REFAZER = False                         # True refaz o que já existe (normalmente: False)


In [ ]:
# @title 2 · Rodadas (8 temporadas + a do Felipe)
RODADAS = S.rodadas(ctx.Du, felipe=True)
D = ctx.Du
pd.DataFrame([dict(rodada=k, grupo=S.grupo_rodada(k), treino_ate=str(pd.Timestamp(D.datas[R['c']]).date()),
                   primeira_semana=str(pd.Timestamp(D.datas[R['t_ini']]).date()),
                   ultima_semana=str(pd.Timestamp(D.datas[R['t_fim']]).date())) for k, R in RODADAS.items()])


## A · Redes de grafo semanais


In [ ]:
# @title A · Lista
TEM_SEL = os.path.exists(ARQ_SEL) and json.load(open(ARQ_SEL)).get('final') is not None
MODELOS = [S.RedeSemanal(U.MTGNN()), S.RedeSemanal(U.GCNGRU(atencao=True)), S.RedeSemanal(U.GCNGRU()),
           S.RedeSemanal(U.STGCN()),
           S.RedeSemanal(U.MTGNN(), H_treino=68),                 # o modelo de temporada, usado para a semana seguinte
           S.RedeSemanal(U.MTGNN(), alvo='cresc'),               # o truque que funcionou no jogo do Felipe
           S.RedeSemanal(U.MTGNN(), cov='todas')]
if TEM_SEL:
    MODELOS.append(S.RedeSemanal(U.MTGNN(), cov='sel', arq_sel=ARQ_SEL))
else:
    print('⚠️ seleção do notebook 1 ainda não existe: MTGNN_sel fica para a próxima execução')
[m.nome for m in MODELOS]


In [ ]:
# @title Rodar (retoma de onde parou; um erro não interrompe o resto)
erros = []
for m in MODELOS:
    erros += S.rodar_semanal(ctx, m, RODADAS, PASTA_SEM, refazer=REFAZER)
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
print('Erros:', len(erros))
pd.DataFrame(erros, columns=['modelo', 'rodada', 'erro']).to_csv(os.path.join(PASTA_SEM, f'erros_{NOTEBOOK}.csv'), index=False)


In [ ]:
# @title Progresso (● = feito)
feitos = {os.path.basename(f)[:-8] for f in glob.glob(os.path.join(PASTA_SEM, '*.parquet'))}
pd.DataFrame({m.nome: {r: '●' if f'{m.nome}__{r}' in feitos else '·' for r in RODADAS} for m in MODELOS}).T


## B · NHITS, TFT e Chronos-2 semanais


In [ ]:
# @title B · Lista
MODELOS = [S.NFSemanal('NHITS'), S.NFSemanal('TFT'), S.NFSemanal('NHITS', cov='todas'), S.NFSemanal('TFT', cov='todas'),
           S.Chronos2Semanal()]
if TEM_SEL:
    MODELOS += [S.NFSemanal('NHITS', cov='sel', arq_sel=ARQ_SEL), S.NFSemanal('TFT', cov='sel', arq_sel=ARQ_SEL)]
[m.nome for m in MODELOS]


In [ ]:
# @title Rodar (retoma de onde parou; um erro não interrompe o resto)
erros = []
for m in MODELOS:
    erros += S.rodar_semanal(ctx, m, RODADAS, PASTA_SEM, refazer=REFAZER)
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
print('Erros:', len(erros))
pd.DataFrame(erros, columns=['modelo', 'rodada', 'erro']).to_csv(os.path.join(PASTA_SEM, f'erros_{NOTEBOOK}.csv'), index=False)


In [ ]:
# @title Progresso (● = feito)
feitos = {os.path.basename(f)[:-8] for f in glob.glob(os.path.join(PASTA_SEM, '*.parquet'))}
pd.DataFrame({m.nome: {r: '●' if f'{m.nome}__{r}' in feitos else '·' for r in RODADAS} for m in MODELOS}).T


## C · Temporada: próximos passos (MTGNN v2)
Mesma régua dos CompUF (dados até a SE25, SE41→SE40). Grava em `resultados_uf_prox/`, que o `CompUF4`, o `Comp5` e o notebook 3 leem junto com as outras pastas `resultados_uf*`.


In [ ]:
# @title C · Rodar (retoma de onde parou)
PROX = {m.nome: m for m in [S.MTGNNv2('cresc'), S.MTGNNv2('misto'), S.MTGNNv2('ancora', peso_pop=0.5)]}
erros = []
for nome, m in PROX.items():
    for s in TUNING + TESTE:
        try:
            dt = L.rodar(ctx, m, s, PASTA_PROX, M=500, refazer=REFAZER)
            if dt is not None:
                print(f'✅ {nome:28s} {s}: {dt / 60:5.1f} min', flush=True)
        except Exception as e:
            erros.append((nome, s, repr(e)[:500]))
            print(f'❌ {nome:28s} {s}: {repr(e)[:300]}', flush=True)
            traceback.print_exc(limit=2)
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
print('Erros:', len(erros))
pd.DataFrame(erros, columns=['modelo', 'temporada', 'erro']).to_csv(os.path.join(PASTA_PROX, 'erros_prox.csv'), index=False)
